# Analysing Players' Performance - Batch Processing - Silver Layer

At the silver level, data starts to take shape. This layer focuses on processing and transforming raw data into a more structured format. Data cleansing, normalization, and integration occur here. The silver layer is critical for ensuring that data is reliable and can be trusted for analysis.

In this layer, we'll be transforming and cleaning the data, including:
- Creating a "Year" column from "Match Date" for ease of filtering and analysis
- Grouping goals by player in each tournament
- Grouping substitutions by player in each tournament
- Grouping bookings (Yellow/Red) by player in each tournament
- Grouping penalty kicks taken by player in each tournament
- Grouping goals by player
- Grouping substitutions by player
- Grouping bookings (Yellow/Red) by player
- Grouping penalty kicks taken by player

In [0]:

# now install the databricks helpers 1
%pip install git+https://github.com/data-derp/databricks_helpers.git


In [0]:
from databricks_helpers.databricks_helpers import DataDerpDatabricksHelpers
exercise_name = "player_analysis_batch_processing/silver"
helpers = DataDerpDatabricksHelpers(dbutils, exercise_name)

current_user = helpers.current_user()
working_directory = helpers.working_directory()
print(f"Your current working directory is: {working_directory}")

## This function CLEARS your current working directory. Only run this if you want a fresh start or if it is the first time you're doing this exercise.
# helpers.clean_working_directory()
bronze_working_directory = working_directory.replace("silver", "bronze")


In [0]:
bronze_output_players = f"{bronze_working_directory}/output/players"
bronze_output_goals = f"{bronze_working_directory}/output/goals"
bronze_output_bookings = f"{bronze_working_directory}/output/bookings"
bronze_output_penalty_kicks = f"{bronze_working_directory}/output/penalty_kicks"
bronze_output_substitutions = f"{bronze_working_directory}/output/substitutions"
brone_output_matches = f"{bronze_working_directory}/output/matches"
player_appearances_bronze = f"{bronze_working_directory}/output/player_appearances"
squads_bronze = f"{bronze_working_directory}/output/squads"

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F

def read_parquet(filepath: str) -> DataFrame:
    df = spark.read.parquet(filepath)
    return df
    
players_bronze_df = read_parquet(bronze_output_players)
goals_bronze_df=read_parquet(bronze_output_goals)
bookings_bronze_df=read_parquet(bronze_output_bookings)
penalty_kicks_bronze_df=read_parquet(bronze_output_penalty_kicks)
substitutions_bronze_df=read_parquet(bronze_output_substitutions)
matches_bronze_df = read_parquet(brone_output_matches)
squads_bronze_df = read_parquet(squads_bronze).select("player_id", "tournament_id", "position_name", "position_code")
player_appearances_bronze_df = read_parquet(player_appearances_bronze)

display(players_bronze_df.limit(5))
display(goals_bronze_df.limit(5))
display(bookings_bronze_df.limit(5))
display(penalty_kicks_bronze_df.limit(5))
display(substitutions_bronze_df.limit(5))
display(squads_bronze_df.limit(5))
display(player_appearances_bronze_df.limit(5))

In [0]:
goals_bronze_df=goals_bronze_df.withColumn("year", F.year("match_date"))
#display(goals_bronze_df.limit(5))
bookings_bronze_df=bookings_bronze_df.withColumn("year", F.year("match_date"))
penalty_kicks_bronze_df=penalty_kicks_bronze_df.withColumn("year", F.year("match_date"))
substitutions_bronze_df=substitutions_bronze_df.withColumn("year", F.year("match_date"))
matches_bronze_df=matches_bronze_df.withColumn("year", F.year("match_date"))

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/matches"
    mode_name = "overwrite"
    input_df.write.mode(mode_name).parquet(out_dir)

write(matches_bronze_df)
dbutils.fs.ls(f"{working_directory}/output/matches")

Join to get number of matches a player played per tournament

In [0]:
def join_with_player_appearances(input_df: DataFrame, column1, column2) -> DataFrame:
    player_appearances_grouped = player_appearances_bronze_df.groupBy(column1, column2).agg(count("match_id").alias("num_matches"))
    return input_df.join(player_appearances_grouped, on=[column1, column2], how="left")

The data below is primed to answer the question:
## How many goals did each player score in each FIFA World Cup tournament?
We group goals by player and tournament to identify the top scorers per competition, spanning every FIFA World Cup from 1930 through to the most recent tournaments.

In [0]:
from pyspark.sql.functions import col, count, countDistinct, sum as spark_sum, when
from pyspark.sql import DataFrame

def goals_per_player(source: DataFrame) -> DataFrame:
    return source \
        .groupBy("player_id", "family_name", "given_name", "team_name", "tournament_id") \
        .agg(count("goal_id").alias("num_goals")) \
        .orderBy(col("num_goals").desc())

goals_per_player_per_tournament_df = goals_bronze_df.transform(goals_per_player)
display(goals_per_player_per_tournament_df.limit(20))
display(goals_per_player_per_tournament_df.count())

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/goals_per_player_per_tournament"
    mode_name = "overwrite"
    input_df.write.mode(mode_name).parquet(out_dir)
    
write(goals_per_player_per_tournament_df)
dbutils.fs.ls(f"{working_directory}/output/goals_per_player_per_tournament")


The data below is primed to answer the question:
## Who are the all-time leading goal scorers in FIFA World Cup history?
We aggregate goals across all tournaments to rank players by their total career World Cup goals, from the 1930 inaugural tournament through to the present day.

In [0]:
def total_goals_per_player(source: DataFrame) -> DataFrame:
    return source \
        .groupBy("player_id", "family_name", "given_name", "team_name") \
        .agg(count("goal_id").alias("num_goals")) \
        .orderBy(col("num_goals").desc())

total_goals_per_player_df = goals_bronze_df.transform(total_goals_per_player)

display(total_goals_per_player_df.limit(20))
display(total_goals_per_player_df.count())

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/goals"
    mode_name = "overwrite"
    input_df.write.mode(mode_name).parquet(out_dir)
    
write(total_goals_per_player_df)
dbutils.fs.ls(f"{working_directory}/output/goals")

The data below is primed to answer the question:
## How many bookings (yellow and red cards) did each player receive in each FIFA World Cup tournament?
We group bookings by player and tournament to identify the most and least disciplined players in each competition.

In [0]:
def bookings_per_player(source: DataFrame) -> DataFrame:
    return source \
        .groupBy("player_id", "family_name", "given_name","team_name","tournament_id") \
        .agg(count("booking_id").alias("num_bookings"),
             spark_sum(when(col("yellow_card") == "1", 1).otherwise(0)).alias("yellow_cards"),
             spark_sum(when(col("red_card") == "1", 1).otherwise(0)).alias("red_cards")
             ).orderBy(col("num_bookings").desc())

bookings_per_player_per_tournament_df = bookings_bronze_df.transform(bookings_per_player)
display(bookings_per_player_per_tournament_df.limit(20))
display(bookings_per_player_per_tournament_df.count())

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/bookings_per_player_per_tournament"
    mode_name = "overwrite"
    input_df.write.mode(mode_name).parquet(out_dir)
    
write(bookings_per_player_per_tournament_df)
dbutils.fs.ls(f"{working_directory}/output/bookings_per_player_per_tournament")

The data below is primed to answer the question:
## Who are the most booked players in FIFA World Cup history?
We aggregate yellow and red card bookings across all tournaments to rank players by total disciplinary actions across their entire World Cup careers.

In [0]:
def total_bookings_per_player(source: DataFrame) -> DataFrame:
    return source \
        .groupBy("player_id", "family_name", "given_name","team_name") \
        .agg(count("booking_id").alias("num_bookings"),
             spark_sum(when(col("yellow_card") == "1", 1).otherwise(0)).alias("yellow_cards"),
             spark_sum(when(col("red_card") == "1", 1).otherwise(0)).alias("red_cards")
             ).orderBy(col("num_bookings").desc())

total_bookings_per_player_df = bookings_bronze_df.transform(total_bookings_per_player)
display(total_bookings_per_player_df.limit(20))
display(total_bookings_per_player_df.count())

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/total_bookings"
    mode_name = "overwrite"
    input_df.write.mode(mode_name).parquet(out_dir)

write(total_bookings_per_player_df)
dbutils.fs.ls(f"{working_directory}/output/total_bookings")

The data below is primed to answer the question:
## How many times was each player substituted on or off in each FIFA World Cup tournament?
We group substitution events by player and tournament to identify players most frequently used as impact substitutes or most often replaced during a competition.

In [0]:
def substitutions_per_player(source: DataFrame) -> DataFrame:
    return source \
        .groupBy("player_id", "family_name", "given_name","team_name","tournament_id") \
        .agg(spark_sum("coming_on").alias("times_subbed_on"),
             spark_sum("going_off").alias("times_subbed_off")
             ).orderBy(col("times_subbed_on").desc())

substitutions_per_player_per_tournament_df = substitutions_bronze_df.transform(substitutions_per_player)
display(substitutions_per_player_per_tournament_df.limit(10))
display(substitutions_per_player_per_tournament_df.count())

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/substitutions_per_player_per_tournament"
    mode_name = "overwrite"
    input_df.write.mode(mode_name).parquet(out_dir)
    
write(substitutions_per_player_per_tournament_df)
dbutils.fs.ls(f"{working_directory}/output/substitutions_per_player_per_tournament")

The data below is primed to answer the question:
## Who are the players most frequently substituted throughout FIFA World Cup history?
We aggregate substitution events across all tournaments to rank players by their total times subbed on and off across their entire World Cup careers.

In [0]:
def total_substitutions_per_player(source: DataFrame) -> DataFrame:
    return source \
        .groupBy("player_id", "family_name", "given_name","team_name") \
        .agg(spark_sum("coming_on").alias("times_subbed_on"),
             spark_sum("going_off").alias("times_subbed_off")
             ).orderBy(col("times_subbed_on").desc())

total_substitutions_per_player_df = substitutions_bronze_df.transform(total_substitutions_per_player)
display(total_substitutions_per_player_df.limit(10))
display(total_substitutions_per_player_df.count())

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/total_substitutions"
    mode_name = "overwrite"
    input_df.write.mode(mode_name).parquet(out_dir)

write(total_substitutions_per_player_df)
dbutils.fs.ls(f"{working_directory}/output/total_substitutions")

In [0]:
def total_penalty_kicks_per_player(source: DataFrame) -> DataFrame:
    return source \
        .groupBy("player_id", "family_name", "given_name","team_name") \
        .agg(
             spark_sum(when(col("converted") == "1", 1).otherwise(0)).alias("penalty_shootout_goals"),count("penalty_kick_id").alias("num_penalty_kicks")
             ).orderBy(col("penalty_shootout_goals").desc())
total_penalty_kicks_per_player_df = penalty_kicks_bronze_df.transform(total_penalty_kicks_per_player)
display(total_penalty_kicks_per_player_df.limit(40))
display(total_penalty_kicks_per_player_df.count())


The data below is primed to answer the question:
## Number of converted penalty shootout goals by each player for each tournament in FIFA World Cup?
We do this by ranking the top 10 players with the most converted goals from penalty shootouts.

In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/total_penalty_kicks"
    mode_name = "overwrite"
    input_df.write.mode(mode_name).parquet(out_dir)

write(total_penalty_kicks_per_player_df)
dbutils.fs.ls(f"{working_directory}/output/total_penalty_kicks")

In [0]:
def penalty_kicks_per_player_per_tournament(source: DataFrame) -> DataFrame:
    return source \
        .groupBy("player_id", "family_name", "given_name","team_name","tournament_id") \
        .agg(
             spark_sum(when(col("converted") == "1", 1).otherwise(0)).alias("penalty_shootout_goals"),count("penalty_kick_id").alias("num_penalty_kicks")
             ).orderBy(col("penalty_shootout_goals").desc())
penalty_kicks_per_player_per_tournament_df = penalty_kicks_bronze_df.transform(penalty_kicks_per_player_per_tournament)
display(penalty_kicks_per_player_per_tournament_df.limit(40))
display(penalty_kicks_per_player_per_tournament_df.count())


In [0]:
def write(input_df: DataFrame):
    out_dir = f"{working_directory}/output/penalty_kicks"
    mode_name = "overwrite"    
    input_df.write.mode(mode_name).parquet(out_dir)
    
write(penalty_kicks_per_player_per_tournament_df)
dbutils.fs.ls(f"{working_directory}/output/penalty_kicks")

In [0]:
def enrich_with_gender(source: DataFrame) -> DataFrame:
    player_gender_df = players_bronze_df.select(
        "player_id",
        F.when(F.col("female") == 1, F.lit("Female"))
         .otherwise(F.lit("Male"))
         .alias("gender")
    )
    return source.join(player_gender_df, "player_id", "left")


def enrich_with_position(source: DataFrame) -> DataFrame:
    squads_position_df = squads_bronze_df.select(
        "player_id", "tournament_id", "position_name", "position_code"
    ).distinct()
    return source.join(squads_position_df, on=["player_id", "tournament_id"], how="left")


In [0]:

goals_per_player_per_tournament_df         = goals_per_player_per_tournament_df.transform(enrich_with_gender)
total_goals_per_player_df                  = total_goals_per_player_df.transform(enrich_with_gender)
bookings_per_player_per_tournament_df      = bookings_per_player_per_tournament_df.transform(enrich_with_gender)
total_bookings_per_player_df               = total_bookings_per_player_df.transform(enrich_with_gender)
substitutions_per_player_per_tournament_df = substitutions_per_player_per_tournament_df.transform(enrich_with_gender)
total_substitutions_per_player_df          = total_substitutions_per_player_df.transform(enrich_with_gender)
penalty_kicks_per_player_per_tournament_df = penalty_kicks_per_player_per_tournament_df.transform(enrich_with_gender)
total_penalty_kicks_per_player_df          = total_penalty_kicks_per_player_df.transform(enrich_with_gender)

print("✅ 'gender' column added to all 8 Silver DataFrames")
display(
    total_goals_per_player_df
    .select("player_id", "family_name", "given_name", "gender", "num_goals")
    .orderBy(F.desc("num_goals"))
    .limit(10)
)

In [0]:
# ── Position enrichment ────────────────────────────────────────────────────────────
# squads gives position_name (e.g. 'Forward') and position_code (e.g. 'FW')
# Join on player_id + tournament_id (per-tournament tables only).
# All-time tables are skipped — a player can hold different positions across tournaments.
goals_per_player_per_tournament_df         = goals_per_player_per_tournament_df.transform(enrich_with_position)
bookings_per_player_per_tournament_df      = bookings_per_player_per_tournament_df.transform(enrich_with_position)
substitutions_per_player_per_tournament_df = substitutions_per_player_per_tournament_df.transform(enrich_with_position)
penalty_kicks_per_player_per_tournament_df = penalty_kicks_per_player_per_tournament_df.transform(enrich_with_position)

print("✅ 'position_name' and 'position_code' added to all 4 per-tournament DataFrames")
for name, df in [
    ("goals_per_player_per_tournament_df",         goals_per_player_per_tournament_df),
    ("bookings_per_player_per_tournament_df",      bookings_per_player_per_tournament_df),
    ("substitutions_per_player_per_tournament_df", substitutions_per_player_per_tournament_df),
    ("penalty_kicks_per_player_per_tournament_df", penalty_kicks_per_player_per_tournament_df),
]:
    nulls = df.filter(F.col("position_name").isNull()).count()
    total = df.count()
    print(f"  {name}: {total - nulls}/{total} rows have position data")

In [0]:
# Re-write the 4 per-tournament silver parquets now that position_name/position_code
# have been joined. This keeps parquet files in sync with UC tables.
for df, path in [
    (goals_per_player_per_tournament_df,          f"{working_directory}/output/goals_per_player_per_tournament"),
    (bookings_per_player_per_tournament_df,        f"{working_directory}/output/bookings_per_player_per_tournament"),
    (substitutions_per_player_per_tournament_df,   f"{working_directory}/output/substitutions_per_player_per_tournament"),
    (penalty_kicks_per_player_per_tournament_df,   f"{working_directory}/output/penalty_kicks"),
]:
    df.write.mode("overwrite").parquet(path)

print("✅ All 4 per-tournament silver parquets rewritten with position_name + position_code")


## Data Quality Checks — Silver Layer
Run these checks after all transforms and before writing to Unity Catalog.
Any failure raises an exception that prevents Cell 33 (UC writes) from executing.

**Checks performed on each DataFrame:**
- `row_count` — DF has at least 1 row
- `no_nulls` — key columns (player_id, tournament_id) contain no nulls
- `no_duplicates` — primary key is unique
- `non_negative` — metric columns have no negative values

In [0]:
from typing import List
import pyspark.sql.functions as F

class DQResult:
    def __init__(self, df_name: str, check: str, passed: bool, detail: str = ""):
        self.df_name  = df_name
        self.check    = check
        self.passed   = passed
        self.detail   = detail
    def __repr__(self):
        status = "\u2705 PASS" if self.passed else "\u274c FAIL"
        return f"{status} | {self.df_name:<30} | {self.check}{' \u2192 ' + self.detail if self.detail else ''}"

def check_row_count(df: DataFrame, df_name: str, min_rows: int = 1) -> DQResult:
    n = df.count()
    return DQResult(df_name, f"row_count >= {min_rows}", n >= min_rows, f"actual: {n:,}")

def check_no_nulls(df: DataFrame, df_name: str, columns: List[str]) -> List[DQResult]:
    results = []
    for c in columns:
        nulls = df.filter(F.col(c).isNull()).count()
        results.append(DQResult(df_name, f"no_nulls({c})", nulls == 0, f"null count: {nulls}"))
    return results

def check_no_duplicates(df: DataFrame, df_name: str, key_cols: List[str]) -> DQResult:
    total    = df.count()
    distinct = df.select(*key_cols).distinct().count()
    dupes    = total - distinct
    return DQResult(df_name, f"no_duplicates({', '.join(key_cols)})", dupes == 0,
                    f"total: {total:,}, distinct: {distinct:,}, dupes: {dupes:,}")

def check_non_negative(df: DataFrame, df_name: str, metric_col: str) -> DQResult:
    neg = df.filter(F.col(metric_col) < 0).count()
    return DQResult(df_name, f"non_negative({metric_col})", neg == 0, f"negative count: {neg}")

In [0]:
dq: List[DQResult] = []

# ── goals_per_player_per_tournament ─────────────────────────────
dq.append(check_row_count(goals_per_player_per_tournament_df, "goals_per_tournament"))
dq.extend(check_no_nulls(goals_per_player_per_tournament_df, "goals_per_tournament", ["player_id", "tournament_id"]))
dq.append(check_no_duplicates(goals_per_player_per_tournament_df, "goals_per_tournament", ["player_id", "tournament_id", "team_name"]))
dq.append(check_non_negative(goals_per_player_per_tournament_df, "goals_per_tournament", "num_goals"))

# ── total_goals_per_player ───────────────────────────────────────
dq.append(check_row_count(total_goals_per_player_df, "total_goals"))
dq.extend(check_no_nulls(total_goals_per_player_df, "total_goals", ["player_id"]))
dq.append(check_no_duplicates(total_goals_per_player_df, "total_goals", ["player_id", "team_name"]))
dq.append(check_non_negative(total_goals_per_player_df, "total_goals", "num_goals"))

# ── bookings_per_player_per_tournament ───────────────────────────
dq.append(check_row_count(bookings_per_player_per_tournament_df, "bookings_per_tournament"))
dq.extend(check_no_nulls(bookings_per_player_per_tournament_df, "bookings_per_tournament", ["player_id", "tournament_id"]))
dq.append(check_no_duplicates(bookings_per_player_per_tournament_df, "bookings_per_tournament", ["player_id", "tournament_id"]))
dq.append(check_non_negative(bookings_per_player_per_tournament_df, "bookings_per_tournament", "num_bookings"))

# ── total_bookings_per_player ────────────────────────────────────
dq.append(check_row_count(total_bookings_per_player_df, "total_bookings"))
dq.extend(check_no_nulls(total_bookings_per_player_df, "total_bookings", ["player_id"]))
dq.append(check_no_duplicates(total_bookings_per_player_df, "total_bookings", ["player_id", "team_name"]))

# ── substitutions_per_player_per_tournament ──────────────────────
dq.append(check_row_count(substitutions_per_player_per_tournament_df, "subs_per_tournament"))
dq.extend(check_no_nulls(substitutions_per_player_per_tournament_df, "subs_per_tournament", ["player_id", "tournament_id"]))
dq.append(check_no_duplicates(substitutions_per_player_per_tournament_df, "subs_per_tournament", ["player_id", "tournament_id"]))

# ── total_substitutions_per_player ───────────────────────────────
dq.append(check_row_count(total_substitutions_per_player_df, "total_subs"))
dq.extend(check_no_nulls(total_substitutions_per_player_df, "total_subs", ["player_id"]))
dq.append(check_no_duplicates(total_substitutions_per_player_df, "total_subs", ["player_id", "team_name"]))

# ── penalty_kicks_per_player_per_tournament ──────────────────────
dq.append(check_row_count(penalty_kicks_per_player_per_tournament_df, "penalties_per_tournament"))
dq.extend(check_no_nulls(penalty_kicks_per_player_per_tournament_df, "penalties_per_tournament", ["player_id", "tournament_id"]))
dq.append(check_no_duplicates(penalty_kicks_per_player_per_tournament_df, "penalties_per_tournament", ["player_id", "tournament_id"]))

# ── total_penalty_kicks_per_player ───────────────────────────────
dq.append(check_row_count(total_penalty_kicks_per_player_df, "total_penalties"))
dq.extend(check_no_nulls(total_penalty_kicks_per_player_df, "total_penalties", ["player_id"]))
dq.append(check_no_duplicates(total_penalty_kicks_per_player_df, "total_penalties", ["player_id"]))

# ── Summary ──────────────────────────────────────────────────────
passed = [r for r in dq if r.passed]
failed = [r for r in dq if not r.passed]

print(f"\n{'='*65}")
print(f"  DATA QUALITY REPORT \u2014 Silver Layer")
print(f"{'='*65}")
print(f"  Total checks : {len(dq)}")
print(f"  \u2705 Passed    : {len(passed)}")
print(f"  \u274c Failed    : {len(failed)}")
print(f"{'='*65}\n")
for r in dq:
    print(r)

if failed:
    raise Exception(
        f"\n\U0001f6a8 {len(failed)} data quality check(s) FAILED. "
        "Fix issues before writing to UC.\n" +
        "\n".join(str(r) for r in failed)
    )
else:
    print(f"\n\u2705 All {len(dq)} checks passed. Safe to proceed with UC writes (Cell 33).")

In [0]:
CATALOG = dbutils.widgets.get("catalog")
if CATALOG != "hive_metastore":
    spark.sql(f"USE CATALOG {CATALOG}")
# spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_NAME}")
SCHEMA_NAME = f"player_analysis"
#Ensure schema exists
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA_NAME}")
matches_bronze_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.matches")
# Goals per player per tournament
goals_per_player_per_tournament_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.goals_per_player_per_tournament")

# Bookings per player per tournament  
bookings_per_player_per_tournament_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.bookings_per_player_per_tournament")

# # Substitutions per player per tournament
substitutions_per_player_per_tournament_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.substitutions_per_player_per_tournament")

# # Penalty kicks per player per tournament (compute all-tournament first)
# from pyspark.sql.functions import col, count, sum as spark_sum, when
# penalty_kicks_all_df = penalty_kicks_bronze_df \
#     .groupBy("player_id", "family_name", "given_name", "team_name", "tournament_id") \
#     .agg(spark_sum(when(col("converted") == "1", 1).otherwise(0)).alias("penalty_shootout_goals"),
#          count("penalty_kick_id").alias("num_penalty_kicks"))

penalty_kicks_per_player_per_tournament_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.penalty_kicks_per_player_per_tournament")
total_goals_per_player_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.total_goals_per_player")
total_substitutions_per_player_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.total_substitutions_per_player")
total_bookings_per_player_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.total_bookings_per_player")
total_penalty_kicks_per_player_df.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fifa_worldcup.player_analysis.total_penalty_kicks_per_player")